# Scraping + Community Analysis + Sentiment Analysis + MindShare

## Dependencies

In [1]:
!pip install macrocosmos dotenv pandas plotly sentence-transformers einops tqdm transformers nbformat

  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached scikit_learn-1.7.2-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonschema-4.25.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (

## Setup

In [16]:
import pandas as pd

from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
import os
import macrocosmos
import plotly.graph_objects as go
import numpy as np
from transformers import pipeline, AutoTokenizer

load_dotenv()

client = macrocosmos.AsyncSn13Client(api_key=os.environ.get("MACROCOSMOS_API_KEY"))

In [4]:
end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=7) # up to minute precision

start_dt.isoformat(), end_dt.isoformat() 

('2025-10-23T08:29:30.713163+00:00', '2025-10-30T08:29:30.713163+00:00')

## Pull Tweets

In [5]:
resp = await client.sn13.OnDemandData(
    source='x', # or 'reddit', 'youtube' -- more examples on following sections
    usernames=None, # None or username(s) ex. ['elonmusk'] or ['elonmusk', 'MikeTyson']
    keywords=["#bitcoin"], # keyword (ex. 'ai'), hashtag (ex. '#bittensor'), or cashtag ('$AAPL')
    start_date=start_dt.date().isoformat(),
    end_date=end_dt.date().isoformat(),  # up to minute precision
    limit=10, # up to 1000
)

f"Response Status: {resp['status']}", f"Data Available: {not resp['meta'].get('no_data_available', False)}"

('Response Status: success', 'Data Available: True')

In [6]:
data = resp.get('data', [])
df = pd.json_normalize(data)

df.head(3)

,text,datetime,uri,source,content_size_bytes,label,tweet.quote_count,tweet.id,tweet.quoted_tweet_id,tweet.retweet_count,...,user.verified,user.id,user.following_count,user.user_location,user.display_name,user.followers_count,user.user_description,user.user_blue_verified,user.profile_image_url,user.username
0,"#Bitcoin will revolutionize payments, much lik...",2025-10-29T23:59:54+00:00,https://x.com/ValueOrion/status/19836852564411...,X,1080.0,#bitcoin,0.0,1983685256441180434,1983624568117010939,0.0,...,False,1847332000954286080,1106.0,None,Hayden⚡️🏴‍☠️,1576.0,"memes, finance, food $GME $₿TC",False,https://pbs.twimg.com/profile_images/192599846...,ValueOrion
1,#Bitcoin $73000 very high probability,2025-10-29T23:59:49+00:00,https://x.com/DopaPrana64/status/1983685231875...,X,961.0,#bitcoin,0.0,1983685231875182999,None,0.0,...,False,1786810155142946818,547.0,None,DopaPrana,637.0,I fear that things will go so wrong going forw...,False,https://pbs.twimg.com/profile_images/178699612...,DopaPrana64
2,Site visit down in South Australia. Early stag...,2025-10-29T23:59:45+00:00,https://x.com/AshurDigital/status/198368521891...,X,1103.0,#bitcoin,0.0,1983685218918969821,None,0.0,...,True,1462205908755173377,11.0,Australia,Ashur Digital Labs,404.0,Out here mining Bitcoin while you scroll. 😜,True,https://pbs.twimg.com/profile_images/195833471...,AshurDigital


In [7]:
df.dtypes

text                           object
datetime                       object
uri                            object
source                         object
content_size_bytes            float64
label                          object
tweet.quote_count             float64
tweet.id                       object
tweet.quoted_tweet_id          object
tweet.retweet_count           float64
tweet.like_count              float64
tweet.conversation_id          object
tweet.hashtags                 object
tweet.reply_count             float64
tweet.is_quote                   bool
tweet.bookmark_count          float64
tweet.is_retweet                 bool
tweet.language                 object
tweet.in_reply_to_user_id      object
tweet.is_reply                   bool
tweet.view_count              float64
tweet.in_reply_to_username     object
user.cover_picture_url         object
user.verified                    bool
user.id                        object
user.following_count          float64
user.user_lo

### Most Influential Users by Followers

In [8]:
print(f"\nTop 5 users by followers:")

# Coerce followers to numeric and group by user (id preferred to dedupe)
df['user.followers_count'] = pd.to_numeric(df['user.followers_count'], errors='coerce').fillna(0)
user_cols = ['user.id','user.display_name','user.username','user.followers_count']

# Drop rows without any user id/username to avoid nonsense groups
users = df[user_cols].copy()

grouped = (users.groupby('user.username', dropna=True)
                .agg({'user.display_name':'last',
                        'user.followers_count':'max'})
                .reset_index()
                .rename(columns={'user.username':'user.username'}))

top_users = grouped.sort_values('user.followers_count', ascending=False).head(5)

for i, row in top_users.reset_index(drop=True).iterrows():
    name = row.get('user.display_name', '')
    uname = row.get('user.username', '')
    followers = int(row['user.followers_count'])
    # Ensure a readable line even if username is missing
    label = f"{name} ({uname})" if uname else name
    print(f"{i+1}. 👤 {label} — {followers} followers")


Top 5 users by followers:
1. 👤 Li₿an_₿itcoin⚡️ (Liban_Bitcoin) — 7736 followers
2. 👤 Corey2140 (Corey2140) — 2218 followers
3. 👤 Hayden⚡️🏴‍☠️ (ValueOrion) — 1576 followers
4. 👤 Trading Maverick (TrMav3rick) — 766 followers
5. 👤 John Niesen ⚡️ ₿ (JohnNiesen) — 733 followers


### Most Liked Tweets

In [9]:
print(f"\nMost 5 liked tweets:")

top_liked = (
    df.sort_values('tweet.like_count', ascending=False)
        .loc[:, ['tweet.like_count','text','user.display_name','user.username','uri']]
        .head(5)
)

# Pretty print
for i, row in top_liked.reset_index().iterrows():
    print(f"{i+1}. ❤️ {int(row['tweet.like_count'])} Likes | {row['user.display_name']} (@{row['user.username']})")
    print(f"   {row['text']}")
    print(f"   {row['uri']}")
    
    print()
    print('=====')
    print()


Most 5 liked tweets:
1. ❤️ 3 Likes | Trading Maverick (@TrMav3rick)
   +1.5R full Tp, #Bitcoin filled 50% of FOMC liquidation wick.
   https://x.com/TrMav3rick/status/1983685077604458848

=====

2. ❤️ 2 Likes | John Niesen ⚡️ ₿ (@JohnNiesen)
   No counterparty risk, always liquid and only 21 million #Bitcoin
   https://x.com/JohnNiesen/status/1983685083715531176

=====

3. ❤️ 1 Likes | Hayden⚡️🏴‍☠️ (@ValueOrion)
   #Bitcoin will revolutionize payments, much like email revolutionized communication.

It’s only a matter of time. 

Stack sats⚡️
   https://x.com/ValueOrion/status/1983685256441180434

=====

4. ❤️ 1 Likes | ℳ𝒶𝓇𝒾𝓊𝓈 - ℐℴ𝒶𝓃 𝒮ℯ𝓇ℯ𝓂ℯ𝓉 〽️ (@MariusSeremet)
   #SamsonMow: The #power of #Bitcoin as #EnergyMoney: 

"It #fuses #energy and #information into the #hardest #form of #money #humanity has ever #created."

"It #requires #energy to #survive and to #propel the entire #timechain."

#BTC
#Binance
#HODL
   https://x.com/MariusSeremet/status/1983685004908708331

=====

5. ❤️ 1 Likes

# Sentiment Analysis

In [ ]:
model_id = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_id)

clf = pipeline(
    "sentiment-analysis",
    model=model_id,
    tokenizer=tokenizer,
)

/Users/ntakouris/code/macrocosmos/sn13-data-universe-api-examples/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps:0


In [11]:
preds = clf(df['text'].astype(str).tolist(), truncation=True, max_length=512, batch_size=32)

In [12]:
df["sentiment"] = [p["label"] for p in preds]
df["sentiment_score"] = [p["score"] for p in preds]

In [ ]:
pos_count = df[df["sentiment"] == "POSITIVE"].shape[0]
neg_count = df[df["sentiment"] == "NEGATIVE"].shape[0]

fig = go.Figure()

# Negative (left)
fig.add_trace(go.Bar(
    x=[-neg_count],
    y=["Sentiment"],
    orientation='h',
    name="Negative",
    marker=dict(color="red")
))

# Positive (right)
fig.add_trace(go.Bar(
    x=[pos_count],
    y=["Sentiment"],
    orientation='h',
    name="Positive",
    marker=dict(color="green")
))

fig.update_layout(
    title="Sentiment Distribution",
    barmode="relative",  # bars diverge from center
    xaxis=dict(title="Count", zeroline=True, zerolinewidth=2),
    showlegend=True
)

fig.show()


## Bullish Tweets

In [22]:
df_pos = df[df["sentiment"] == "POSITIVE"].copy()

# Compute bullish score: sentiment * log(views + 1)
df_pos["bullish_score"] = df_pos["sentiment_score"] * np.log1p(df_pos["tweet.view_count"])

df_bullish = df_pos.sort_values("bullish_score", ascending=False)

df_bullish_display = df_bullish[
    ["text", "sentiment_score", "tweet.view_count", "bullish_score", "datetime", "user.username"]
].head(10)

for _, row in df_bullish_display.iterrows():
    print(f'- {row['text']}, views={row['tweet.view_count']}')

- Site visit down in South Australia. Early stages now, but big things ahead! 🚧 

#bitcoin #cryptomining, views=7.0


# Community Analysis

### Mindshare

In [23]:
df["tweet.view_count"] = df["tweet.view_count"].fillna(0)

# Group by username
mindshare = (
    df.groupby("user.username")["tweet.view_count"]
    .sum()
    .reset_index(name="total_views")
)

# mindshare(user) = sum(user_views) / sum(all_views)
total_views_sum = mindshare["total_views"].sum()
mindshare["mindshare_pct"] = mindshare["total_views"] / total_views_sum * 100

mindshare = mindshare.sort_values("mindshare_pct", ascending=False)

mindshare.head(15)  # top 15 accounts by share of attention

,user.username,total_views,mindshare_pct
8,TrMav3rick,207.0,31.603053
9,ValueOrion,132.0,20.152672
5,Liban_Bitcoin,115.0,17.557252
4,JohnNiesen,59.0,9.007634
1,Corey2140,56.0,8.549618
7,THE_8_TRADER,31.0,4.732824
6,MariusSeremet,18.0,2.748092
2,DopaPrana64,15.0,2.290076
3,Follow_BTC_News,15.0,2.290076
0,AshurDigital,7.0,1.068702
